# Two real decisions. One program-earnings prediction system.
## A business-friendly inference demonstration | BUAN 6341 — Group 2

**The simple promise:** A prospective student tells us **which bachelor's program at which college** they are considering. Instead of answering with a generic national salary headline, our system looks up the college and labor-market context, passes **exactly 31 known fields** to our best trained model, and responds in plain English.

### How to present
1. **Run the setup cell once.** It loads the previously saved model and the model-ready dataset. **There is no training or hyperparameter tuning in this notebook.**
2. **Run Scenario 1.** A Computer Science applicant wants a realistic planning figure for a specific public university.
3. **Run Scenario 2.** A family asks whether a specific public university Nursing program deserves a place on their shortlist.

Each scenario's **single code cell** narrates: the person's question → the information they give us → the extra context our system retrieves → **the exact 31 features the model actually receives** → the saved model's answer → what we would say back to the person → a separate historical reality check.

**Our production-style forecast is the Random Forest**, which achieved **$6,232 average absolute error** and **7.6% median percentage error** across the held-out test set. Gradient Boosting is shown only as a second-model check. The program's actual earnings are **never given to either model**.

**Fairness to the audience:** These two genuine records were *specifically selected* from the held-out test data because the models performed especially well on them. They illustrate the **workflow and interpretation**, not the typical precision of every prediction. Both schools were excluded from model training and tuning.

**What the prediction means:** A historical estimate of *program-level median earnings among federally aided graduates, four years after completion*. **Not** an individual person's guaranteed salary, causal effect of school choice, admissions decision, or a fully calculated return on investment.

In [1]:
# SETUP — read our existing artifacts; no fitting, tuning, charts, or plots.
from pathlib import Path
import joblib
import numpy as np
import pandas as pd

def project_root():
    for folder in [Path.cwd(), *Path.cwd().parents,
                   Path.home() / "Projects/classes/buan-6341/college-majors-2026-earnings-debt-ml"]:
        if (folder / "3. Modelling/saved_models/random_forest_pipeline.joblib").is_file():
            return folder
    raise FileNotFoundError("Run this notebook inside the college majors project.")

ROOT = project_root()
MODEL_DIR = ROOT / "3. Modelling" / "saved_models"
rf_model = joblib.load(MODEL_DIR / "random_forest_pipeline.joblib")
gb_model = joblib.load(MODEL_DIR / "gradient_boosting_pipeline.joblib")
FEATURES = list(rf_model.named_steps["prep"].feature_names_in_)
assert FEATURES == list(gb_model.named_steps["prep"].feature_names_in_)
assert len(FEATURES) == 31

records = pd.read_csv(ROOT / "2. EDA" / "bachelors_model_ready.csv",
                      dtype={"opeid6": str, "cip_code_4digit": str})
test = records.loc[records["split"].eq("test")].copy()
train_school_ids = set(records.loc[records["split"].eq("train"), "opeid6"])
assert train_school_ids.isdisjoint(set(test["opeid6"]))
assert not set(FEATURES).intersection({
    "median_earnings_4yr_usd", "target_log_earnings_4yr",
    "target_above_national", "log_median_debt", "program_id"
})

# For transparency, show every single input our stored pipeline receives.
# The display groups only organize the printed explanation; model column order
# always follows FEATURES, not the groups below.
INPUT_GROUPS = [
    ("The program and its delivery", [
        "log_awards_avg", "is_main_campus", "outcomes_shared_across_campuses",
        "cip_family_title", "online"]),
    ("The school, its location, tuition and admissions context", [
        "log_enrollment", "institution_control", "institution_region",
        "institution_tuition_in_state_usd", "tuition_out_minus_in",
        "institution_admission_rate", "admission_rate_missing",
        "institution_avg_sat", "sat_missing", "institution_is_hbcu",
        "institution_latitude", "institution_longitude", "selectivity"]),
    ("The national field benchmark and related job market", [
        "national_median_earnings_4yr_usd", "linked_occupations_count",
        "occupation_median_wage_2024_usd", "occupation_growth_pct_2024_34",
        "occupation_growth_pct_max", "log_occ_employment",
        "occupation_typical_entry_education"]),
    ("Technology exposure in associated occupations", [
        "occupations_using_ai_software", "ai_software_occupation_share",
        "expert_system_occupation_share", "ai_tools_max_per_occupation",
        "hot_technologies_mean_per_occupation", "any_ai_occupation"]),
]
flat_keys = [key for _, keys in INPUT_GROUPS for key in keys]
assert len(flat_keys) == len(set(flat_keys)) == 31 and set(flat_keys) == set(FEATURES)

def find_program(program_id):
    record = test.loc[test["program_id"].eq(program_id)]
    assert len(record) == 1, f"Expected one held-out record for {program_id}"
    assert record.iloc[0]["opeid6"] not in train_school_ids
    return record

def show_exact_input(record):
    print("WHAT THE SYSTEM SENDS TO THE MODEL — ALL 31 ORIGINAL INPUT FIELDS")
    print("The person does NOT have to enter these 31 values manually.")
    print("Our system retrieves them from the verified program/institution/job dataset.")
    print("Missing values are preserved as missing; the saved pipeline handles them.")
    for group_name, keys in INPUT_GROUPS:
        print(f"\n   {group_name.upper()}:")
        for key in keys:
            raw = record.iloc[0][key]
            if isinstance(raw, np.generic):
                raw = raw.item()
            print(f"      {key:<45} = {raw!r}")
    print("\n   The pipeline receives this exact ordered 31-column row:")
    print("   [", ", ".join(FEATURES), "]")
    print("   The historical program earnings, debt-to-earnings and future outcomes are EXCLUDED.")

def get_forecast(record):
    features_only = record.loc[:, FEATURES].copy()
    assert features_only.shape == (1, 31)
    return (float(np.exp(rf_model.predict(features_only)[0])),
            float(np.exp(gb_model.predict(features_only)[0])))

def dollars(x):
    return f"\u0024{x:,.0f}"

def check_against_history(record, random_forest, gradient_boosting):
    actual = float(record.iloc[0]["median_earnings_4yr_usd"])
    print(f"   Recorded historical median for this program : {dollars(actual)}")
    print(f"   Random Forest's distance from actual         : {dollars(abs(random_forest - actual))} ({abs(random_forest - actual) / actual:.1%})")
    print(f"   Gradient Boosting's distance from actual     : {dollars(abs(gradient_boosting - actual))} ({abs(gradient_boosting - actual) / actual:.1%})")
    assert np.isfinite(actual)
    assert abs(random_forest - actual) / actual < .05
    assert abs(gradient_boosting - actual) / actual < .07
    print("   These are intentionally selected GOOD examples, not typical model accuracy.")

print("SETUP COMPLETE")
print("Best model: Saved Random Forest (Gradient Boosting used as independent comparison)")
print(f"Test data: {len(test):,} bachelor's programs at {test['opeid6'].nunique():,} previously unseen schools")
print("Ready for the two stakeholder stories. No model retraining required.")


SETUP COMPLETE
Best model: Saved Random Forest (Gradient Boosting used as independent comparison)
Test data: 5,351 bachelor's programs at 360 previously unseen schools
Ready for the two stakeholder stories. No model retraining required.


## SCENARIO 1 — A Computer Science applicant plans their finances

**The person:** A prospective bachelor's student considering **Western Michigan University** in Michigan.

**Their concern:** “I keep hearing that Computer Science graduates earn about $107,000. Is that the right figure to plan around if I enroll at this university?”

**What they tell our application:** *School = Western Michigan University; Program = Bachelor's in Computer Science; Goal = understand a realistic four-year-after-graduation earnings benchmark.*

**What our application looks up:** The school's public/private status and location; its institution-wide admission rate and tuition; the program's field, scale and delivery; and associated labor-market and AI-exposure measures. The student does **not** need to understand or enter all of those variables. Our data already contains them.

**What the next code cell demonstrates:** Every one of the 31 features the trained model receives, the saved Random Forest forecast, the normal-language response the student receives, and a historical result shown **only afterward** to check the example.

**The decision we help with:** Avoid anchoring a financial plan to the **national Computer Science median** when this university's estimated program-level outcome differs.

In [2]:
# SCENARIO 1 — A spoken, end-to-end customer interaction
program = find_program("172699-11.07-3")
school = program.iloc[0]["institution_name"]
field_reference = float(program.iloc[0]["national_median_earnings_4yr_usd"])

print("=" * 96)
print("STORY 1  |  STUDENT: 'CAN I EXPECT THE NATIONAL COMPUTER SCIENCE SALARY HERE?'")
print("=" * 96)
print("\nACT 1 — THE QUESTION FROM THE PERSON")
print('   "I am considering Computer Science at Western Michigan University.')
print('    Everyone quotes a national CS earnings figure. What should I realistically plan for?"')

print("\nACT 2 — THE ONLY DETAILS THE PERSON HAS TO PROVIDE")
print(f"   School                     : {school}")
print("   Degree and field           : Bachelor's degree, Computer Science")
print("   Their decision             : Estimate the typical program outcome before enrolling.")
print("   They do not need to know the school's internal dataset measurements.")

print("\nACT 3 — OUR SYSTEM FINDS THE PROGRAM AND PREPARES THE MODEL INPUT")
print("   School/program identity selects one existing record. Its attributes are")
print("   checked against the saved preprocessing pipeline's feature schema.")
show_exact_input(program)

print("\nACT 4 — WE PASS ONLY THOSE INPUTS TO OUR ALREADY TRAINED MODELS")
rf_1, gb_1 = get_forecast(program)
print(f"   Random Forest prediction    : {dollars(rf_1)} median annual earnings four years after completion")
print(f"   Gradient Boosting check     : {dollars(gb_1)}")
print(f"   National CS reference       : {dollars(field_reference)}")
print("   This is an inference prediction, NOT a retrained model or a simple lookup of actual earnings.")

print("\nACT 5 — EXACTLY HOW WE WOULD ANSWER THE PERSON")
print(f'   "For the Computer Science program at {school},')
print(f'    our model estimates that graduates typically earn around {dollars(rf_1)}')
print('    a year, four years after finishing the degree.')
print(f'    That is approximately {dollars(field_reference-rf_1)} below the national CS reference')
print(f'    of {dollars(field_reference)}. So rather than budgeting around a generic')
print('    CS headline, use this school-specific figure as a more relevant starting')
print('    point. Also compare tuition, scholarships, career services and outcomes')
print('    for other colleges. This is a program-level estimate, not a promise')
print('    about your own future salary."')

print("\nACT 6 — AFTER THE DEMO, WE OPEN THE HELD-OUT ANSWER TO CHECK THE FORECAST")
check_against_history(program, rf_1, gb_1)

print("\nTHE STAKEHOLDER TAKEAWAY")
print("   The major tells only part of the story. Our model gives the student")
print("   a school-specific planning benchmark before making a costly choice.")


STORY 1  |  STUDENT: 'CAN I EXPECT THE NATIONAL COMPUTER SCIENCE SALARY HERE?'

ACT 1 — THE QUESTION FROM THE PERSON
   "I am considering Computer Science at Western Michigan University.
    Everyone quotes a national CS earnings figure. What should I realistically plan for?"

ACT 2 — THE ONLY DETAILS THE PERSON HAS TO PROVIDE
   School                     : Western Michigan University
   Degree and field           : Bachelor's degree, Computer Science
   Their decision             : Estimate the typical program outcome before enrolling.
   They do not need to know the school's internal dataset measurements.

ACT 3 — OUR SYSTEM FINDS THE PROGRAM AND PREPARES THE MODEL INPUT
   School/program identity selects one existing record. Its attributes are
   checked against the saved preprocessing pipeline's feature schema.
WHAT THE SYSTEM SENDS TO THE MODEL — ALL 31 ORIGINAL INPUT FIELDS
The person does NOT have to enter these 31 values manually.
Our system retrieves them from the verified pr

## SCENARIO 2 — A family decides whether to shortlist a Nursing program

**The people:** A student and parent comparing bachelor's Nursing programs at public universities.

**Their concern:** “We are considering **California State University–Fresno**. The nationwide Nursing earnings median is below $90,000, but our family's planning goal is a program whose typical graduates earn at least $100,000. Is Fresno worth a closer look?”

**What they tell our application:** *School = California State University–Fresno; Program = Bachelor's in Registered Nursing; Planning goal = $100,000 in median earnings four years after graduation.*

**What our application looks up:** The program and institution's 31 recorded features: in-state tuition, location, selectivity information when available, field-wide reference earnings, local occupation wages and job-growth indicators, among others. In this example, **$7,350 in-state tuition** and **95% institution-wide admission** are recorded context; **95% is NOT the Nursing program's admission rate**.

**The decision we help with:** Identify a program whose **school-specific earnings outlook is stronger** than the national Nursing average. We do **not** imply that tuition or institutional admissions causes this outcome, and we do not claim that the prediction measures net ROI.

In [3]:
# SCENARIO 2 — A family asks if a nursing program merits a closer look
program = find_program("110556-51.38-3")
school = program.iloc[0]["institution_name"]
field_reference = float(program.iloc[0]["national_median_earnings_4yr_usd"])
planning_goal = 100_000

print("=" * 96)
print("STORY 2  |  FAMILY: 'SHOULD FRESNO STATE NURSING GO ON OUR SHORTLIST?'")
print("=" * 96)
print("\nACT 1 — THE QUESTION FROM THE PERSON")
print('   "We are looking at public-university Nursing programs.')
print('    The national median is about $89,000, but our planning target is $100,000.')
print('    Could the Fresno nursing program be a promising option?"')

print("\nACT 2 — THE ONLY DETAILS THE FAMILY HAS TO PROVIDE")
print(f"   School                     : {school}")
print("   Degree and field           : Bachelor's degree, Registered Nursing")
print(f"   Their planning goal        : {dollars(planning_goal)} annual median earnings four years after completion")
print("   Our application retrieves the rest; they do not enter 31 fields.")

print("\nACT 3 — OUR SYSTEM FINDS THE EXACT PROGRAM RECORD AND CONTEXT")
print("   It extracts known school, program and occupation features from the")
print("   model-ready dataset, including local context and the national Nursing reference.")
print(f"   Recorded in-state tuition  : {dollars(float(program.iloc[0]['institution_tuition_in_state_usd']))} per year")
print("   Institution-wide admission : 95% (NOT the Nursing program's admission rate)")
show_exact_input(program)

print("\nACT 4 — WE RUN THE TRAINED MODEL WITH THAT EXACT RECORD")
rf_2, gb_2 = get_forecast(program)
print(f"   Random Forest prediction    : {dollars(rf_2)} median annual earnings four years after completion")
print(f"   Gradient Boosting check     : {dollars(gb_2)}")
print(f"   National Nursing reference  : {dollars(field_reference)}")
print(f"   Above the family's goal?    : {'YES' if rf_2 >= planning_goal else 'NO'}")

print("\nACT 5 — EXACTLY HOW WE WOULD ANSWER THE FAMILY")
print(f'   "For the Registered Nursing program at {school},')
print(f'    our model estimates that graduates typically earn around {dollars(rf_2)}')
print('    a year, four years after completing the program.')
print(f'    That is about {dollars(rf_2-field_reference)} above the nationwide Nursing median')
print(f'    of {dollars(field_reference)}, and it clears your {dollars(planning_goal)} planning goal.')
print('    It therefore deserves a place on your shortlist for career outcomes.')
print('    Before deciding, check program-specific admission criteria, aid,')
print('    tuition, living expenses and licensure requirements separately.')
print('    This is a model estimate for the typical program, not a guaranteed')
print('    salary or proof that one college causes higher earnings."')

print("\nACT 6 — AFTER THE DEMO, WE REVEAL THE HELD-OUT HISTORICAL RESULT")
check_against_history(program, rf_2, gb_2)

print("\nTHE STAKEHOLDER TAKEAWAY")
print("   A national salary average can hide promising college-specific outcomes.")
print("   Our model helps a family find an option worth deeper investigation.")


STORY 2  |  FAMILY: 'SHOULD FRESNO STATE NURSING GO ON OUR SHORTLIST?'

ACT 1 — THE QUESTION FROM THE PERSON
   "We are looking at public-university Nursing programs.
    The national median is about $89,000, but our planning target is $100,000.
    Could the Fresno nursing program be a promising option?"

ACT 2 — THE ONLY DETAILS THE FAMILY HAS TO PROVIDE
   School                     : California State University-Fresno
   Degree and field           : Bachelor's degree, Registered Nursing
   Their planning goal        : $100,000 annual median earnings four years after completion
   Our application retrieves the rest; they do not enter 31 fields.

ACT 3 — OUR SYSTEM FINDS THE EXACT PROGRAM RECORD AND CONTEXT
   It extracts known school, program and occupation features from the
   model-ready dataset, including local context and the national Nursing reference.
   Recorded in-state tuition  : $7,350 per year
   Institution-wide admission : 95% (NOT the Nursing program's admission rate)
